In [1]:
import torch
from torch.cuda import amp
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [9]:
model = nn.Linear(10, 2).cuda()
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [10]:
# Criando dados fictícios
images = torch.randn(8, 10).cuda()
labels = torch.randint(0, 2, (8,)).cuda()

# Configurando DataLoader
dataset = TensorDataset(images, labels)
dataloader = DataLoader(dataset, batch_size=2)
scaler = torch.amp.GradScaler('cuda')
accum_steps = 4

In [11]:
for i, (batch_images, batch_labels) in enumerate(dataloader):
  with torch.amp.autocast('cuda'):
    outputs = model(batch_images)
    loss = criterion(outputs, batch_labels) / accum_steps

  scaler.scale(loss).backward()

  if (i + 1) % accum_steps == 0:
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)